In [ ]:
import glob
import json
import os

import torch

os.chdir("..")
os.getcwd()

In [ ]:
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
os.environ["DATA_DIR"]

# s2bms

In [ ]:
uc = "s2bms"
uc = "satbird-USA-summer"

output_dir = f"{os.environ["DATA_DIR"]}/{uc}/aux_stats/"
os.makedirs(output_dir, exist_ok=True)

lst = glob.glob(f"{os.environ["DATA_DIR"]}/{uc}/model*.csv")
lst

In [ ]:
for item in lst:
    # same name as csv file
    output_path = output_dir + item.split("/")[-1].replace(".csv", ".json")
    if os.path.exists(output_path):
        print(f"Aux stats  {output_path} already exists.")
        continue

    df = pd.read_csv(item)

    # Split train filter
    if uc == "s2bms":
        if "unlabelled" in item:
            pth = "data/s2bms/splits/s2bms_unlabelled_union_val_test.pth"
        else:
            pth = "data/s2bms/splits/s2bms_union_val_test.pth"
    elif uc == "satbird-USA-summer":
        pth = pth = "data/satbird-USA-summer/splits/satbird-USA-summer_aef_union_val_test_lc.pth"
        # pth = pth =  "data/satbird-USA-summer/splits/satbird-USA-summer_aef_union_val_test.pth"

    split_indices = torch.load(pth, weights_only=False)
    df = df[df["name_loc"].isin(split_indices["train_indices"])]

    stats = {}

    cols = [i for i in df.columns if "aux" in i and "top" not in i]

    for col in cols:
        stats[col] = {
            "mean": float(df[col].mean()),
            "std": float(df[col].std()),
        }

    with open(output_path, "w") as f:
        json.dump(stats, f, indent=4)

    print(f"Saved global statistics for {len(stats)} variables to {output_path}")